# AlphaStar + OpenAI Five Clash Royale Agent
### Hybrid Google Colab Tesla T4 (16GB VRAM) & Local RTX 3050 Pipeline

This notebook runs the PPO+GAE training loop with:
- **AlphaStar Entity Transformer + Spatial CNN + Autoregressive Action Heads**
- **OpenAI Five Clipped PPO + GAE ($\\gamma=0.99, \\lambda=0.95$)**
- **Prioritized Fictitious Self-Play (PFSP) with Historical League Pool**
- **Action Repeat = 4 (Frame-Skip ~400ms cadence, 4x FPS boost)**
- **Class-Weighted Imitation Warm-Start (prevents No-Op collapse)**
- **FP16 Mixed Precision & Dynamic Gradient Accumulation**
- **Seamless Checkpoint Sync via Google Drive (`latest.pt`)**

## 1. Verify GPU Hardware (Tesla T4 Expected)

In [ ]:
!nvidia-smi

## 2. Mount Google Drive for Continuous Checkpointing
Checkpoints are saved to `/content/drive/MyDrive/clashbot/checkpoints` so progress is never lost when Colab disconnects.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/clashbot/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"[Ready] Checkpoints will sync with: {CKPT_DIR}")

## 3. Clone or Copy Codebase and Install Dependencies

In [ ]:
# If using Google Drive storage for repo:
# %cd /content/drive/MyDrive/clashbot

# Or install required packages
!pip install -q matplotlib numpy
import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB VRAM)")

## 4. (Optional) Run Behavioral Cloning Warm-Start
Run 500 matches of bot-vs-bot supervised learning with class weighting `[5.0, 5.0, 5.0, 5.0, 0.25]` to seed aggressive tower defense and card deployment.

In [ ]:
# Full 500-match supervised warmstart
!python warmstart_bc.py --num-matches 500 --epochs 5 --checkpoint-dir /content/drive/MyDrive/clashbot/checkpoints
# Or add --smoke-test for a quick 10-match verification

## 5. Launch OpenAI Five + AlphaStar Self-Play Training Loop
Automatically detects the 16GB Tesla T4, sets `micro_batch=256` (1 pass), and auto-resumes from `/content/drive/MyDrive/clashbot/checkpoints/latest.pt`.
Runs full production batches (4 envs x 512 steps = 2048 samples per update).

In [ ]:
!python train.py --checkpoint-dir /content/drive/MyDrive/clashbot/checkpoints --total-updates 500 --num-envs 8 --rollout-steps 256
# Or add --smoke-test for a quick 2-update verification

## 6. Inspect Saved Checkpoints in Google Drive

In [ ]:
!ls -lh /content/drive/MyDrive/clashbot/checkpoints
!ls -lh /content/drive/MyDrive/clashbot/checkpoints/pool

## 7. Run AlphaStar League Elo Tournament & Multi-Agent Benchmarks
Runs round-robin matches with canonical symmetry across model checkpoints, pool snapshots, and baselines (ScriptedBot, RushExploiterBot, RandomBot).

In [ ]:
# Run AlphaStar League Elo Tournament across all checkpoints and pool snapshots
!python tools/league_tournament.py --games-per-pair 2 --max-candidates 6

# Evaluate against Scripted Bot
!python eval_agent.py --games 20 --opponent scripted

# Evaluate against Rush / Bridge-Spam Exploiter Bot
!python eval_agent.py --games 10 --opponent rush